# 18장 전이학습 — 남이 학습한 것 위에서 시작하기

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`, `p02` 를 치고 실행. `p01` 의 `named_children()` 출력을 17장 코드와 대조 | [ch18_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `p02` 의 A·B·C 각각에 "학습되는 파라미터 / 얼린 파라미터 / 학습률"을 주석으로 | [ch18_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_2_comment.ipynb) |
| 3. 다시 만들기 | 14장 패키지에 `--model resnet18 --pretrained --freeze` 인자를 추가해 B 방식이 되게 하기 | [ch18_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "18"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/18")

준비 끝. 작업 폴더: code/18


## `p01_pretrained.py` — 18.1 남이 학습한 모델 불러오기: torchvision 의 ResNet-18 (ImageNet 사전학습)

처음 실행하면 사전학습 가중치(약 45MB)를 내려받습니다.

In [ ]:
# 18.1 남이 학습한 모델 불러오기: torchvision 의 ResNet-18 (ImageNet 사전학습)
import torch
from torchvision import models

weights = models.ResNet18_Weights.DEFAULT              # 사전학습 가중치 지정 (처음 한 번 내려받음, 약 45MB)
model = models.resnet18(weights=weights)
n = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 파라미터 {n:,}")
print("마지막 층:", model.fc)                              # Linear(512 → 1000): ImageNet 의 1000 클래스

ResNet-18 파라미터 11,689,512
마지막 층: Linear(in_features=512, out_features=1000, bias=True)


In [ ]:
# 어떤 층들이 있나: 이름만 (17장의 stem / stages / head 와 대응)
for name, child in model.named_children():
    print(f"  {name:<8} {child.__class__.__name__}")

  conv1    Conv2d
  bn1      BatchNorm2d
  relu     ReLU
  maxpool  MaxPool2d
  layer1   Sequential
  layer2   Sequential
  layer3   Sequential
  layer4   Sequential
  avgpool  AdaptiveAvgPool2d
  fc       Linear


In [ ]:
# 사전학습 모델이 요구하는 전처리 (크기·정규화) 도 가중치와 함께 온다
print("전처리:", weights.transforms())

전처리: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
# 가짜 이미지 하나를 넣어 본다
x = torch.zeros(1, 3, 224, 224)
model.eval()
with torch.no_grad():
    out = model(x)
print("출력:", tuple(out.shape), "| 상위 3 클래스 번호:", out[0].topk(3).indices.tolist())
print("클래스 이름 예:", [weights.meta["categories"][i] for i in out[0].topk(3).indices.tolist()])

출력: (1, 1000) | 상위 3 클래스 번호: [623, 111, 499]
클래스 이름 예: ['letter opener', 'nematode', 'cleaver']


## `p02_finetune.py` — 18.2 전이학습 세 방식 비교 (도형 데이터, ResNet-18)

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 18.2 전이학습 세 방식 비교 (도형 데이터, ResNet-18)
#   A. 처음부터 학습 (사전학습 없음)
#   B. 특징 추출: 사전학습 가중치를 얼리고 마지막 층만 학습
#   C. 미세조정: 전부 학습하되 작은 학습률
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "15"))
from p02_custom_dataset import ImageFolderDataset

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")
# 사전학습 모델의 전처리에 맞춘다: 크기 224 는 CPU 에서 느리므로 112 로 줄여 쓴다 (ResNet 은 크기에 유연)
tf = transforms.Compose([transforms.Resize((112, 112)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])   # ImageNet 통계
train_set = ImageFolderDataset(os.path.join(ROOT, "train"), tf)
test_set = ImageFolderDataset(os.path.join(ROOT, "test"), tf)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)
NUM_CLASSES = len(train_set.classes)


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_set)


def make_model(pretrained):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)      # 마지막 층을 내 문제(3 클래스)용으로 교체
    return m


def run(name, model, params, lr, epochs=2):
    torch.manual_seed(0)
    opt = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    for _ in range(epochs):
        for x, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
    n_train = sum(p.numel() for p in params)
    print(f"{name:<22} 학습 파라미터 {n_train:>10,}  정확도 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")


if __name__ == "__main__":
    # A
    torch.manual_seed(0)
    m = make_model(pretrained=False)
    run("A. 처음부터", m, list(m.parameters()), lr=1e-3)
    # B
    torch.manual_seed(0)
    m = make_model(pretrained=True)
    for p in m.parameters():
        p.requires_grad = False                            # 얼리기
    for p in m.fc.parameters():
        p.requires_grad = True                             # 마지막 층만 학습
    run("B. 특징 추출(fc만)", m, list(m.fc.parameters()), lr=1e-3)
    # C
    torch.manual_seed(0)
    m = make_model(pretrained=True)
    run("C. 미세조정(전부)", m, list(m.parameters()), lr=1e-4)
```

In [ ]:
!python p02_finetune.py

A. 처음부터                학습 파라미터 11,178,051  정확도 0.680  (18초)
B. 특징 추출(fc만)          학습 파라미터      1,539  정확도 1.000  (7초)
C. 미세조정(전부)            학습 파라미터 11,178,051  정확도 1.000  (18초)


## 다음 단계

- 2단계 「주석 달기」: [ch18_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch18_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.